In [ ]:
# %load ../../notebooks/init.ipy
%reload_ext autoreload
%autoreload 2

# Builtin packages
from datetime import datetime
from importlib import reload
import logging
import os
from pathlib import Path
import sys
import warnings

# standard secondary packages
import astropy as ap
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import scipy.stats
import tqdm.notebook as tqdm


# development packages
import kalepy as kale
import kalepy.utils
import kalepy.plot

# --- Holodeck ----
import holodeck as holo
from holodeck import cosmo, utils, plot
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC, NWTG, SCHW
import holodeck.sams
import holodeck.gravwaves
#import holodeck.evolution
#import holodeck.population

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode
plt.rcParams.update({'grid.alpha': 0.5})

# Load log and set logging level
log = holo.log
# log.setLevel(logging.INFO)
#log.setLevel(logging.DEBUG)
#log.setLevel(log.DEBUG)
#log.level, log.DEBUG, log.INFO


# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
#_SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #

#SPEED_LIMIT = 0.1 * SPLC


In [ ]:
def get_sam_dadt(nrads=100, shape=None, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                 ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5,
                 calc_gwb=False, nreals=10, nloud=1):
    if gpf_flag: 
        _gpf = sams.GPF_Power_Law()
    else:
        _gpf = None
    
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
        
        
    sam = holo.sams.Semi_Analytic_Model(
        mtot = (1.0e4*MSOL, 1.0e12*MSOL, 91),
        #mtot = (1.0e5*MSOL, 1.0e11*MSOL, 91),
        #mtot = (1.0e9*MSOL, 1.0e10*MSOL, 91),
        shape = shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    hard = holo.hardening.Fixed_Time_2PL_SAM(
        sam, tau * GYR,
        sepa_init = ainit * PC,
        rchar = rc * PC,
        gamma_inner = nu_in,
        gamma_outer = nu_out
    )
    #print("checking status of class hard")
    #print(hard)
    #print(dir(hard))
    #print(vars(hard))
    #print(issubclass(hard,holo.hardening._Hardening))

    print(f"before defining radii: {sam.mtot.shape=} {sam.mrat.shape=} {sam.redz.shape=}")
    # () start from the hardening model's initial separation
    rmax = hard._sepa_init
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    # rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )
    # (X, M*Q*Z)
    #mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    #print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    #print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')
    #print('Mtot=', sam.mtot/MSOL)
    #print('q=', sam.mrat)
    #print('redz=', sam.redz)

    # old: (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    # new: (M, Q, Z, X) is shape of input and output arrays
    dadt = hard.dadt(mt, mr, rads)
    print(f"{mt.shape=}, {mr.shape=}, {dadt.shape=}, {rads.shape=}")

    if calc_gwb:
        freqs, freqs_edges = utils.pta_freqs()
        gwb = sam.gwb(freqs_edges, hard, realize=nreals, loudest=nloud, params=True)
        # returns hc_ss, hc_bg, sspar, bgpar as tuple
        return sam, hard, rads, dadt, gwb
    else:
        return sam, hard, rads, dadt

    
def get_sam_newhard_dadt(nrads=100, shape=None, gsmf_flag=2, gpf_flag=0,
                         tau_outer=1.0, rc=100, nu_in=-1.0, 
                         dadt_rc=None, gwc_units='rg', 
                         rgw9=1e3, alphagw=0, 
                         in_time=None,
                         in_mod_type=0,
                         calc_gwb=False, nreals=10, nloud=1):
    if gpf_flag: 
        _gpf = sams.GPF_Power_Law()
    else:
        _gpf = None
    
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
        
        
    sam = holo.sams.Semi_Analytic_Model(
        mtot = (1.0e4*MSOL, 1.0e12*MSOL, 91),
        #mtot = (1.0e5*MSOL, 1.0e11*MSOL, 91),
        #mtot = (1.0e9*MSOL, 1.0e10*MSOL, 91),
        shape = shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    #OLD:
    #newhard = holo.hardening.FixedOuterTime_InnerPL_SAM(
    #    sam, 
    #    tau_outer * GYR,
    #    rchar = rc * PC,
    #    gamma_inner = nu_in,
    #    x_gw_crit = xcrit,
    #    num_steps = nrads
    #)   
    #NEW:
    #self, sam, num_steps=300, outer_time=1.0*GYR, rchar=100.0*PC, 
    #             gamma_inner=-1.0, x_gw_crit=1e3, gw_crit_units='rg',
    #             r_gw_crit_9=0.01*PC, alpha_gw_crit=0.75, dadt_rchar=None, 
    #             inner_time=None,
    #             inner_model_type=0):
    print(f"{nrads=}")
    newhard = holo.hardening.FixedOuterTime_InnerPL_SAM(
        sam, 
        num_steps = nrads,
        outer_time = tau_outer * GYR,
        rchar = rc * PC,
        gamma_inner = nu_in,
        gw_crit_units = gwc_units,
        r_gw_crit_9 = rgw9, alpha_gw_crit = alphagw, 
        dadt_rchar=dadt_rc,
        inner_time = in_time,
        inner_model_type=in_mod_type
    )
    #print("checking status of class newhard")
    #print(newhard)
    #print(dir(newhard))
    #print(vars(newhard))
    #print(issubclass(newhard,holo.hardening._Hardening))

    # () start from the hardening model's initial separation
    rmax = newhard._rchar
    print(f"{newhard._rchar=} {newhard._rchar/PC=}")
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )
    # (X, M*Q*Z)
    #mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    #print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    #print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')
    #print('Mtot=', sam.mtot/MSOL)
    #print('q=', sam.mrat)
    #print('redz=', sam.redz)

    # old: (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    # new: (M, Q, Z, X) is shape of input and output arrays
    dadt, agw_crit, rz_char, rz_final = newhard.dadt(mt, mr, rz, rads)
    print(f"{mt.shape=}, {mr.shape=}, {rz.shape=}, {dadt.shape=}, {rads.shape=}")
    print(f"{dadt.shape=}, {rz_char.shape=}, {rz_final.shape=}")
    print(f"*** {rads.max()=} {rads.min()=} ***")
    if calc_gwb:
        freqs, freqs_edges = utils.pta_freqs()
        gwb = sam.gwb(freqs_edges, hard=newhard, realize=nreals, loudest=nloud, params=True)
        # returns hc_ss, hc_bg, sspar, bgpar as tuple
        
        return sam, newhard, rads, dadt, agw_crit, rz_char, rz_final, gwb
    else:
        return sam, newhard, rads, dadt, agw_crit, rz_char, rz_final
    

def sepa_emit(mtot, fgw):
    """
    separation of an equal-mass circular binary with total mass mtot emitting GWs at frequency fgw
    
    assumes mtot in cgs and fgw in Hz, returns separation in cm
    """
    #print(f'{MSOL=}, {mtot=}, {fgw=}, {NWTG=}')
    return ( NWTG * mtot / (fgw * np.pi) **2 )**(1.0/3)

def calc_and_plot_dadt(sam_data, distance_units='pc', fixedTime='total', 
                       max_to_plot = 4, extra_panels=False, verbose=False):
    
    #if gpf_flag: 
    #    _gpf = sams.GPF_Power_Law()
    #else:
    #    _gpf = None
    
    #if gsmf_flag == 1:
    #    _gsmf = holo.sams.GSMF_Schechter()
    #elif gsmf_flag == 2:
    #    _gsmf = holo.sams.GSMF_Double_Schechter()
    #else: 
    #    raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
           
    #sam = holo.sams.Semi_Analytic_Model(
    #    shape=shape,
    #    gsmf = _gsmf,
    #    gpf = _gpf
    #)
    #hard = holo.hardening.Fixed_Time_2PL_SAM(
    #    sam, tau * GYR,
    #    sepa_init = ainit * PC,
    #    rchar = rc * PC,
    #    gamma_inner = nu_in,
    #    gamma_outer = nu_out
    #)
    
    ## () start from the hardening model's initial separation
    #rmax = hard._sepa_init
    ## (M,) end at the ISCO
    #rmin = utils.rad_isco(sam.mtot)
    ## rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
    ## Choose steps for each binary, log-spaced between rmin and rmax
    #extr = np.log10([rmax * np.ones_like(rmin), rmin])
    #radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    ## (M, X)
    #radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    #radii = 10.0 ** radii
    ## (M, Q, Z, X)
    #mt, mr, rz, rads = np.broadcast_arrays(
    #    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    #    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    #    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    #    radii[:, np.newaxis, np.newaxis, :]
    #)
    # (X, M*Q*Z)
    ##mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    ##print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    ##print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')

    ## (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    #dadt = hard.dadt(mt, mr, rads)
    #print(f"{dadt.shape=}")

    
    if distance_units == 'pc':
        xlim=[1e-8,1e5]
    elif distance_units == 'rg':
        xlim=[1,1e13]
    else:
        raise ValueError(f"invalid keyword {distance_units=}. must be 'pc' or 'rg'.")
    
    if fixedTime not in ['total','outer']:
        raise ValueError(f"keyword `fixedTime` must be 'total' or 'outer'.")
    
    # Make the plot
    if extra_panels:
        fig = plt.figure(figsize=(12,9))
        first_plot_index = 231
    else:
        fig = plt.figure(figsize=(12,4))
        first_plot_index = 131
        
    
    ax1 = fig.add_subplot(first_plot_index)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax1.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('hardening tscale [yr]')
    #plt.ylim(1e4,1e10)

    ax2 = fig.add_subplot(first_plot_index+1)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax2.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('hardening rate [cm/s]')
    #plt.ylim(1e4,1e10)

    ax3 = fig.add_subplot(first_plot_index+2)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax3.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('cumulative time [yr]')

    if extra_panels:
        ax4 = fig.add_subplot(first_plot_index+3)
        plt.xscale('log')
        plt.xlim(xlim[0],xlim[1])
        plt.yscale('log')
        ax4.xaxis.set_inverted(True)
        plt.xlabel(f'binary separation [{distance_units}]')
        plt.ylabel(r's (hardening parameter) ')

        ax5 = fig.add_subplot(first_plot_index+4)
        plt.xscale('log')
        #plt.xlim(xlim[0],xlim[1])
        plt.yscale('log')
        #ax5.xaxis.set_inverted(True)
        plt.xlabel(r'<s_inner> [pc Myr]$^{-1}$')
        plt.ylabel('cumulative time [yr]')

        #ax4 = fig.add_subplot(first_plot_index+3)
        #plt.xscale('log')
        #plt.xlim(xlim[0],xlim[1])
        #plt.yscale('log')
        #ax4.xaxis.set_inverted(True)
        #plt.xlabel(f'a_GW,crit [{distance_units}]')
        #plt.ylabel('time from fobs=1/30yr to ISCO [yr]')

        #ax5 = fig.add_subplot(first_plot_index+4)
        #plt.xscale('log')
        #plt.xlim(xlim[0],xlim[1])
        #plt.yscale('log')
        #ax5.xaxis.set_inverted(True)
        #plt.xlabel(f'a_GW,crit [{distance_units}]')
        #plt.ylabel('a(fobs=1/30yr) / a_GW,crit')

    
    cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
                'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*10
    fgw_ls = ['-','-.',':']
    lhandles = []
    flhandles = []
    
    nu_mrk = ['s','^','o','*']
    
    for n,sd in enumerate(sam_data):
        
        if fixedTime=='total':
            if len(sd) == 4:
                sam, hard, rads, dadt = sd
            elif len(sd) == 5:
                sam, hard, rads, dadt, gwb = sd
            else:
                raise ValueError(f"sam_data has unexpected length {len(sd)}. must be 4 or 5 for `fixedTime`='total'.")
                
            agw = calc_aGW_for_Fixed_Time_2PL(hard, sam)
            #print(f"{hard._norm.shape=} {agw.shape=}")
            
        else: 
            # fixedTime == 'outer'
            if len(sd) == 7:
                sam, hard, rads, dadt, agw, rzch, rzf = sd
            elif len(sd) == 8:
                sam, hard, rads, dadt, agw, rzch, rzf, gwb = sd
            else:
                raise ValueError(f"sam_data has unexpected length {len(sd)}. must be 7 or 8 for `fixedTime`='outer'")

        #if np.abs(dadt).max() > SPEED_LIMIT:
        #    #print(f"***WARNING!***\n{dadt.shape=}, {np.abs(dadt).max()=}, {SPEED_LIMIT=}, "
        #    #      f"{hard._gamma_inner=}, {hard._r_gw_crit_9=}, {hard._alpha_gw_crit=}")
        #    print(f"***WARNING!*** max dadt/limit={np.abs(dadt).max()/SPEED_LIMIT:.4g}, "
        #          f"nu_in={hard._gamma_inner}, r9={hard._r_gw_crit_9}, alph={hard._alpha_gw_crit}")
        #    continue
        #else:
        #    print(f"VALID PARAMS: nu_in={hard._gamma_inner}, r9={hard._r_gw_crit_9}, alph={hard._alpha_gw_crit}")
                          
        #print(f"{sam.mtot.shape=}, {sam.mrat.shape=}")
        mt_nskip = int((sam.mtot.size-1)/(max_to_plot-1)) if sam.mtot.size>max_to_plot else 1
        mr_nskip = int((sam.mrat.size-1)/(max_to_plot-1)) if sam.mrat.size>max_to_plot else 1

        
        times_evo = calc_cumulative_thard(sd, rads[0,0,0,0], rads[0,0,0,-1],fixedTime=fixedTime)
        #times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
         
        cmap = plot._get_cmap(cmap_arr[n])
        colors = cmap(np.linspace(0.3, 1, max_to_plot+1))
        lw = np.arange(0.5,max_to_plot+1, 0.5)

        freqs, freqs_edges = utils.pta_freqs()

        if verbose:
            print(f"*** in plotting function: {rads.min()=} {rads.max()=}")
            print(f"{mt_nskip=}, {mr_nskip=}")
            print(f"{rads[0,0,0,0]=}, {rads[0,0,0,-1]=}")
            print(freqs_edges.min(),freqs_edges.max())
            if n==0:
                print('Mtot=', sam.mtot/MSOL)
                print('q=', sam.mrat)
                print('redz=', sam.redz)

        
        i_plot = 0
        for i in np.arange(0,sam.mtot.size,mt_nskip):
            
            if distance_units == 'pc':
                dunits = PC
            elif distance_units == 'rg':
                dunits = NWTG * sam.mtot[i] / SPLC**2
            
            #print(f'Mtot = {sam.mtot[i]/MSOL:.2g}')

            #for fi,frst in enumerate([freqs_edges.min(),freqs_edges.max()]):
            frst_min = utils.frst_from_fobs(freqs_edges.min(), sam.redz.min())
            sepa_obs_max = sepa_emit(sam.mtot[i],frst_min) / dunits
            flmi,= ax1.plot([sepa_obs_max,sepa_obs_max],[1e-4,1e10],ls='-',
                            alpha=0.7,color=colors[i_plot],label=f'frst={frst_min:.2g}Hz')
            if i_plot==max_to_plot and n==len(sam_data)-1:
                flhandles += [flmi]
                    
            #for fi,fem in enumerate([0.1,0.3,1.0]):
            #    sepa_em = sepa_emit(sam.mtot[i], fem/YR) / dunits
            #    fl,= ax1.plot([sepa_em,sepa_em],[1e-4,1e10],ls=fgw_ls[fi],
            #                  alpha=0.5,color=colors[i],label=f'fgw={fem}/yr')
            #    #ax2.plot([sepa_em,sepa_em],[1e-3,1e11],ls=fgw_ls[fi],color=colors[i])
            #    #print(f'{fi=},{fgw_ls[fi]}, {fem}')
            #    if i==sam.mtot.size-1 and n==len(sam_data)-1:
            #        flhandles += [fl]
                    
            a_late = sepa_emit(sam.mtot[i], 1.0/(30*YR)) 
            #print(f"mtot={sam.mtot[i]/MSOL:.3g}: in {distance_units}: {a_late/dunits=:.3g} "
            #      f"{agw[i,0]/dunits=:.3g} {agw[i,-1]/dunits=:.3g} {a_late/agw[i,0]=:.3g} {a_late/agw[i,-1]=:.3g}")
            #print(f"{sam.mtot[i]=}: {a_late/dunits=} [{distance_units}]")
            times_early = calc_cumulative_thard(sd, rads[0,0,0,0], a_late,fixedTime=fixedTime)
            times_late = calc_cumulative_thard(sd, a_late, rads[0,0,0,-1],fixedTime=fixedTime)
            #print(f"{times_evo.shape=},{times_early.shape=},{times_late.shape=}")
            ax2.plot([a_late/dunits, a_late/dunits], [1e-3,1e11], color=colors[i_plot], alpha=0.1)

            j_plot=0
            for j in np.arange(0,sam.mrat.size,mr_nskip):
                
                if fixedTime=='total':
                    l,= ax1.plot(rads[i,j,0,:]/dunits, -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot], 
                                 label=f'tau={hard._target_time/GYR}')
                    if i_plot==max_to_plot and j_plot==max_to_plot:
                        lhandles += [l]
                else: 
                    ax1.plot(rads[i,j,0,:]/dunits, -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot])

                if j_plot==max_to_plot and n==len(sam_data)-1:
                    ax2.plot(rads[i,j,0,:]/dunits, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot], 
                             label=f'mtot={sam.mtot[i]/MSOL:.2g}')
                else:
                    ax2.plot(rads[i,j,0,:]/dunits, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i_plot], lw=lw[j_plot],label=None)
                #print(f"plotting {agw[i,j]=} for mtot={sam.mtot[i]/MSOL}, mrat={sam.mrat[j]}")
                #ax2.plot([agw[i,j]/dunits, agw[i,j]/dunits], [1e-3,1e11], color=colors[i], alpha=0.1)
                
                
                ax3.plot(rads[i,j,0,:-1]/dunits, times_evo[i,j,:]/YR, 
                         alpha=0.5, color=colors[i_plot], lw=lw[j_plot])
                #print(f"{times_early[i,j,:].size=}, {times_late[i,j,:].size=}")
                #nt_early = times_early[i,j,:].size
                #nt_late = times_late[i,j,:].size
                #ax3.plot(rads[i,j,0,:nt_early]/dunits, times_early[i,j,:]/YR, 
                #         alpha=0.5, color='r', lw=lw[j])
                #ax3.plot(rads[i,j,0,nt_early+1:-1]/dunits, (times_early[i,j,-1]+times_late[i,j,:])/YR, 
                #         alpha=0.5, color='b', lw=lw[j])
                
                j_plot += 1

                if extra_panels:

                    hard_param_s = -dadt[i,j,0,:]/rads[i,j,0,:]**2*PC*YR*1.0e6 # 1/(pc*Myr)
                    avg_hard_param_s = hard_param_s.mean()
                    #print(f"{hard_param_s.shape=} {hard_param_s.min()=} {hard_param_s.max()=} {avg_hard_param_s=}")
                
                    if j_plot==max_to_plot and n==len(sam_data)-1:
                        ax4.plot(rads[i,j,0,:]/dunits, hard_param_s, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot],
                                 label=f'mtot={sam.mtot[i]/MSOL:.2g}')
                    else:
                        ax4.plot(rads[i,j,0,:]/dunits, hard_param_s, 
                                 alpha=0.5, color=colors[i_plot], lw=lw[j_plot],label=None)
                    ax5.scatter(avg_hard_param_s, times_evo[i,j,-1]/YR)

            i_plot += 1
        
        if fixedTime=='total':
            ax1.plot(xlim, [hard._target_time/YR, hard._target_time/YR], '--', color='darkgray')
            if distance_units=='pc': ax1.plot([hard._rchar/dunits, hard._rchar/dunits], [1e-2,1e10],'k--')
            if distance_units=='pc': ax2.plot([hard._rchar/dunits, hard._rchar/dunits], [10,1e11],'k--')
    ax2.plot(xlim, [3.0e10,3.0e10], color='magenta')
    ax3.plot(xlim, [13.7e9,13.7e9], 'k:')
    leg2 = ax1.legend(handles=flhandles, loc='lower right')
    #ax1.legend(handles=lhandles, loc='lower left')
    ax1.add_artist(leg2)
    ax2.legend(loc='lower left')
        
    if fixedTime=='total':
        plt.suptitle(f'ai={hard._sepa_init/PC:.2g}pc, '
                     f'rc={hard._rchar/PC:.2g}pc,'
                     f' nu_in={hard._gamma_inner}, nu_out={hard._gamma_outer}\n'
                     f'Mtot=({sam.mtot.min()/MSOL:.2g},{sam.mtot.max()/MSOL:.2g})Msun, '
                     f'q=({sam.mrat.min():.2g},{sam.mrat.max():.2g})')
    fig.subplots_adjust(wspace=0.3,top=0.85, right=0.95)
    

In [ ]:
from compare_sams import load_sams_from_pkl

In [ ]:
def calc_sam_dadt_from_pkl(sam, nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                           num_steps=num_steps, verbose=False):

    # () start from the hardening model's initial separation
    rmax = sam.hard._rchar
    # (M,) end at the ISCO
    rmin = utils.rad_isco(s.sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, num_steps)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )

    if verbose:
        print(sam.model_type)
        print(f"{sam.PARS['hard_outer_time']=}")
        print(sam.sam.mtot.shape, sam.sam.mrat.shape, sam.sam.redz.shape)
        print(sam.gwb_sam[0].shape,sam.gwb_sam[1].shape,sam.gwb_sam[2].shape,sam.gwb_sam[3].shape)
        print(f"{s.hard._rchar=} {sam.hard._rchar/PC=}")
        print(rmin.shape, rmin.min(), rmin.max())
        print(np.log10(sam.hard._rchar), np.log10(rmin), num_steps)

    dadt, agw_crit, rz_char, rz_final = s.hard.dadt(mt, mr, rz, rads)

    gpf_flags = [0]*len(sams_newhard_typ0_toutvar) 

    return sam.sam, sam.hard, rads, dadt, agw_crit, rz_char, rz_final, sam.gwb_sam


In [ ]:
freqs, freqs_edges = utils.pta_freqs()
zmin=1.0e-5
zmax=10
fomin=freqs_edges.min()
fomax=freqs_edges.max()
Mtot=1e9*MSOL
for z in [zmin,zmax]:
    for fo in [fomin,fomax]:
        for M in [1e4*MSOL,1e8*MSOL, 1e12*MSOL]:
            fr=utils.frst_from_fobs(fo,z)
            sp=sepa_emit(M,fr)
            print(f"{fo=:.4g}, {z=}, {M/MSOL=:.4g}, frst={fr:.4g}, "
                  f"sep={sp/PC:.4g}pc ={sp/(NWTG*M/SPLC**2):.4g}rg")
#print(f"{fomax=:.6g}, z={zmax}, frst={utils.frst_from_fobs(fomax,zmax):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomax,zmax))/PC:.6g}")
#print(f"{fomin=:.6g}, z={zmin}, frst={utils.frst_from_fobs(fomin,zmin):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomin,zmin))/PC:.6g}")
#print(f"{fomax=:.6g}, z={zmin}, frst={utils.frst_from_fobs(fomax,zmin):.6g}, sep={sepa_emit(Mtot,utils.frst_from_fobs(fomax,zmin))/PC:.6g}")


#fobs_min = utils.fobs_from_frst(freqs_edges.min(), sam.redz.max())
#fobs_max = utils.fobs_from_frst(freqs_edges.max(), sam.redz.min())
print(f"{1/(fomin*YR)=:.4g} yr")

In [ ]:
NLOUD = 5
NREALS = 10
NFREQS = 40
num_steps = 100
sams_newhard_typ0_toutvar = load_sams_from_pkl(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                                               gpf_flag=False, tau=None, data_dir=_SIM_MERGER_PATH, 
                                               fname_type='new_hardening_type0_toutvar')

dadt_sams_newhard_typ0_toutvar = []
for s in sams_newhard_typ0_toutvar: 
    dadt_sams_newhard_typ0_toutvar = ( dadt_sams_newhard_typ0_toutvar + 
                                       [(calc_sam_dadt_from_pkl(s, nloud=NLOUD, nreals=NREALS, 
                                                                nfreqs=NFREQS, num_steps=num_steps))] )



In [ ]:
calc_and_plot_dadt(dadt_sams_newhard_typ0_toutvar, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(dadt_sams_newhard_typ0_toutvar, fixedTime='outer', distance_units='rg', extra_panels=False)

In [ ]:
def calc_cumulative_thard(_sam_data, astart, astop, fixedTime='total'):

    if fixedTime not in ['total','outer']:
        raise ValueError(f"keyword `fixedTime` must be 'total' or 'outer'.")

    if fixedTime=='total':
            if len(_sam_data) == 4:
                _sam, _hard, _rads, _dadt = _sam_data
            elif len(_sam_data) == 5:
                _sam, _hard, _rads, _dadt, _gwb = _sam_data
            else:
                raise ValueError(f"sam_data has unexpected length {len(_sam_data)}. must be 4 or 5 for `fixedTime`='total'.")
    else:
        if len(_sam_data) == 7:
            _sam, _hard, _rads, _dadt, _agw_crit, _rz_char, _rz_final = _sam_data
        elif len(_sam_data) == 8:
            _sam, _hard, _rads, _dadt, _agw_crit, _rz_char, _rz_final, _gwb = _sam_data
        else:
            raise ValueError(f"sam_data has unexpected length {len(_sam_data)}. must be 7 or 8 for `fixedTime`='outer'")


    idx_avals = np.where((_rads[0,0,0,:]<=astart)&(_rads[0,0,0,:]>=astop))[0]
    
    _tevo = -utils.trapz_loglog(-1.0 / _dadt[:,:,0,idx_avals], _rads[:,:,0,idx_avals], 
                                axis=2, cumsum=True)
    return _tevo


def calc_aGW_for_Fixed_Time_2PL(_hard, _sam):
    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""

    
    _mt, _mr = np.broadcast_arrays(
        _sam.mtot[:, np.newaxis],
        _sam.mrat[np.newaxis, :]
    )
    
    _m1, _m2 = utils.m1m2_from_mtmr(_mt, _mr)
    #print(f"{mt/MSOL=},{mr=}")
    #print(f"{m1/MSOL=},{m2/MSOL=}")
    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * _m1 * _m2 * _mt
    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
    dadt_innerPL_const = - _hard._norm * _hard._rchar**(_hard._gamma_inner-1)
    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
    
    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-_hard._gamma_inner))
    #print(f"{aGW=}")
    
    return aGW

#def calc_aGW_for_Fixed_Time_2PL(norm, nu_inner, rchar, mtot, mrat):
#    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""
#
#    
#    mt, mr = np.broadcast_arrays(
#        mtot[:, np.newaxis],
#        mrat[np.newaxis, :]
#    )
#    
#    m1, m2 = utils.m1m2_from_mtmr(mt, mr)
#    #print(f"{mt/MSOL=},{mr=}")
#    #print(f"{m1/MSOL=},{m2/MSOL=}")
#    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * m1 * m2 * mt
#    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
#    dadt_innerPL_const = - norm * rchar**(nu_inner-1)
#    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
#    
#    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-nu_inner))
#    #print(f"{aGW=}")
#    
#    return aGW

In [ ]:
nr = 100
sh = 10
gsmff = 2
gpff = 0
ai = 1.0e4
rch=10.0
t=1.0
nin=0.0
nout=2.5
sam, hard, rads, dadt = get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                     ainit=ai, tau=t, rc=rch, nu_in=nin, nu_out=nout)

print(hard._norm.shape, sam.mtot.shape, sam.mrat.shape)
print(hard._norm, sam.mtot/MSOL, sam.mrat)
#agw_test = calc_aGW_for_Fixed_Time_2PL(hard._norm, hard._gamma_inner, hard._rchar, sam.mtot, sam.mrat)
agw_test = calc_aGW_for_Fixed_Time_2PL(hard, sam)
print(agw_test.shape)
print(agw_test/PC)

times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
print(f"{dadt.shape=}, {rads.shape=}, {times_evo.shape=}")
tt = times_evo/GYR
#tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='log')
ax.xaxis.set_inverted(True)
print(utils.stats(tt))
#kale.dist1d(tt, density=True)
for m in range(rads.shape[0]):
    for q in range(rads.shape[1]):
        plt.plot(rads[m,q,0,:-1]/PC,tt[m,q,:])
plt.show()

In [ ]:
print(agw_test.shape, sam.mtot.shape, sam.mrat.shape)
plt.xscale('log')
plt.yscale('log')
for i in range(sam.mrat.size):
    plt.scatter(sam.mtot/MSOL,agw_test[:,i]/PC)
    plt.plot(sam.mtot/MSOL,agw_test[:,i]/PC)

In [ ]:
alphaGW = np.arange(0,1.5,0.01)
dadt_index = 3*(1-alphaGW)
tscale_index = 4*alphaGW - 3
plt.plot(alphaGW, alphaGW,label='aGW')
plt.plot(alphaGW, dadt_index,label='dadt_GW')
plt.plot(alphaGW, tscale_index,label='tscale_GW')
plt.plot([0.75,0.75],[-3,3],'k:')
plt.plot([6/7,6/7],[-3,3],'k:')
plt.plot([1,1],[-3,3],'k:')
plt.plot([0,1.5],[0,0],'k:')
plt.plot([0,1.5],[1,1],'k:')
plt.legend()

In [ ]:
#calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
#                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5)

nr = 100
sh = 100
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
_calc_gwb = True
#for t in [0.1,1.0,10.0]:
for t in [1.0]:
    for rch in [100]:
        for nin in [-1.5]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                                     nreals=nr, calc_gwb=_calc_gwb))]

print(len(sam_data))
print(len(sam_data[0]))
##calc_and_plot_dadt(sam_data)
#calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total')
#calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {freqs.shape=}")
    fig = plot.plot_gwb(freqs, hc_bg) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))


In [ ]:
calc_and_plot_dadt(sam_data)

In [ ]:
calc_and_plot_dadt(sam_data, distance_units='rg')

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
alph = 0.0
_dadt_rc = None
_in_mod_type = 0
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
#print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 1.0
nin = None
#xcr = 1000.0
#_gwc_units = 'rg'
r9 = 30
alph = -0.25
_gwc_units = 'rg'
_dadt_rc = -10.0**3.5 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)

In [ ]:
#calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
#                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5)

nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
_calc_gwb = True
#for t in [0.1,1.0,10.0]:
t = 1.0
rch = 10
nin = -1.0
nout = 2.5

if _calc_gwb:
    sam, hard, rads, dadt, gwb = get_sam_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, 
                                              gpf_flag=gpff, ainit=ai, 
                                              tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                              nreals=nreal, calc_gwb=_calc_gwb)
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {freqs.shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 ) 
    fig = plot.plot_gwb(freqs, hc_tot)
else:
    sam, hard, rads, dadt = get_sam_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, 
                                         gpf_flag=gpff, ainit=ai, 
                                         tau=t, rc=rch, nu_in=nin, nu_out=nout,
                                         nreals=nreal, calc_gwb=_calc_gwb)
##calc_and_plot_dadt(sam_data)
#calc_and_plot_dadt(sam_data, distance_units='rg',fixedTime='total')
#calc_and_plot_dadt(sam_data, distance_units='pc',fixedTime='total')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')


In [ ]:
nrad = 100
nreal = 100
sh = 100
gsmff = 2
gpff = 0
tau_out = 10.0
rch = 100.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = True

if _calc_gwb:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf, gwb 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                             tau_outer=tau_out, rc=rch, nu_in=nin, 
                                             rgw9=r9, alphagw=alph,
                                             dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                             in_mod_type=_in_mod_type,
                                             nreals=nreal, calc_gwb=_calc_gwb),)

    print(len(sam_newhard_data[0]))
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 )
    print("hc_bg=",hc_bg)
    fig = plot.plot_gwb(freqs, hc_tot)


else:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb),)

##calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
##calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')

#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
#calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)



In [ ]:
nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = True

if _calc_gwb:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf, gwb 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                             tau_outer=tau_out, rc=rch, nu_in=nin, 
                                             rgw9=r9, alphagw=alph,
                                             dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                             in_mod_type=_in_mod_type,
                                             nreals=nreal, calc_gwb=_calc_gwb),)

    #print(len(sam_newhard_data[0]))
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    #print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    #print(f"{freqs.shape=} {hc_bg[:,0].shape=}")
    hc_tot = np.sqrt( np.sum(hc_ss**2, axis=-1) + hc_bg**2 )
    #print("hc_bg=",hc_bg)
    fig = plot.plot_gwb(freqs, hc_tot)


else:
    #sam, newhard, rads, dadt, agw_cr, rzch, rzf 
    sam_newhard_data = (get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb),)

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False)

In [ ]:
nrad = 100
nreal = 10
sh = 4
gsmff = 2
gpff = 0
tau_out = 1.0
rch = 100.0
#rch = 1.0
nin = -1.0
r9 = 1000.0
_gwc_units = 'rg'
#r9 = 0.05
#_gwc_units = 'pc'
alph = -0.25
_dadt_rc=None
#_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 0
_calc_gwb = False
sam_newhard_data = []

nin_arr = np.array([-1.5, -1.0, -0.5, 0.0, 0.5])
alph_arr = np.array([-0.5,-0.333,-0.25,-0.1,0.0])
r9_arr = np.array([30,100,300,1000,3000])
valid_arr = np.zeros((nin_arr.size,alph_arr.size,r9_arr.size)).astype('int')
print(valid_arr.shape)
for i,nin in enumerate(nin_arr):
    for j,alph in enumerate(alph_arr):
        for k,r9 in enumerate(r9_arr):
            tmp = get_sam_newhard_dadt(nrads=nrad, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                            tau_outer=tau_out, rc=rch, nu_in=nin, 
                                            rgw9=r9, alphagw=alph,
                                            dadt_rc=_dadt_rc, gwc_units=_gwc_units, 
                                            in_mod_type=_in_mod_type,
                                            nreals=nreal, calc_gwb=_calc_gwb)
            print(len(tmp))
            if np.abs(tmp[3]).max() <= SPEED_LIMIT:
                valid_arr[i,j,k] = 1
            sam_newhard_data = sam_newhard_data + [(tmp)]

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=False)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=False)



In [ ]:
#fig, axs = plt.subplots(2,3)
#print(valid_arr)
mtyp=['.','+','o','^','s']
col=['darkorange','m','g','b','k']
plt.xlabel('log10(r9)')
plt.ylabel('alpha')
for i,n in enumerate(nin_arr):
    for j,al in enumerate(alph_arr):
        for k,r9 in enumerate(r9_arr):
            if valid_arr[i,j,k]:
                if j==0 and k==r9_arr.size-1:
                    plt.scatter(np.log10(r9_arr[k])+0.05*i,alph_arr[j],marker=mtyp[i],color=col[i],label=f"nuin={n}")
                else:
                    plt.scatter(np.log10(r9_arr[k])+0.05*i,alph_arr[j],marker=mtyp[i],color=col[i])
plt.legend()
plt.suptitle(f'param combos for which all dadt<{SPEED_LIMIT/SPLC}c (rch={rch}pc)')

In [ ]:
print(np.vstack([nin_arr,alph_arr,r9_arr]).shape,valid_arr.shape)
data = np.vstack([nin_arr[1:2],alph_arr,r9_arr]).T
print(valid_arr.T.flatten().shape)
figure = corner.corner(data)

In [ ]:
# Set up the parameters of the problem.
ndim, nsamples = 3, 50000

# Generate some fake data.
np.random.seed(42)
data1 = np.random.randn(ndim * 4 * nsamples // 5).reshape(
    [4 * nsamples // 5, ndim]
)
data2 = 4 * np.random.rand(ndim)[None, :] + np.random.randn(
    ndim * nsamples // 5
).reshape([nsamples // 5, ndim])
data = np.vstack([data1, data2])

# Plot it.
figure = corner.corner(
    data,
    labels=[
        r"$x$",
        r"$y$",
        r"$\log \alpha$",
        r"$\Gamma \, [\mathrm{parsec}]$",
    ],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 12},
)

print(data.shape, data.min(), data.max())
print(data1.shape, data1.min(), data1.max())
print(data2.shape, data2.min(), data2.max())

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
tau_out = 10.0
rch = 100.0
#nin = -0.5
nin = None
#r9 = 1000.0
#_gwc_units = 'rg'
r9 = 0.05
_gwc_units = 'pc'
alph = -1
#_dadt_rc=None
_dadt_rc = -1.0e7 #cm/s
_in_mod_type = 1
_calc_gwb = False
    
sam_newhard_data = (get_sam_newhard_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                          tau_outer=tau_out, rc=rch, nu_in=nin, 
                                          rgw9=r9, alphagw=alph,
                                          dadt_rc=_dadt_rc, gwc_units=_gwc_units, in_mod_type=_in_mod_type,
                                          nreals=nr, calc_gwb=_calc_gwb),)

print(len(sam_newhard_data))
print(len(sam_newhard_data[0]))
print(len(sam_newhard_data[0][-1]))
#calc_and_plot_dadt(sam_newhard_data, distance_units='rg',fixedTime='outer')
#calc_and_plot_dadt(sam_newhard_data, distance_units='pc',fixedTime='outer')
if _calc_gwb:
    gwb = sam_newhard_data[0][-1]
    hc_ss, hc_bg, sspar, bgpar = gwb
    freqs, freqs_edges = utils.pta_freqs()
    print(f"{hc_ss.shape=} {hc_bg.shape=} {sspar.shape=}, {bgpar.shape=} {freqs.shape=}")
    print(f"{freqs[:-1].shape=} {hc_bg[:,0].shape=}")
    plt.xscale('log')
    plt.yscale('log')
    fig = plt.plot(freqs[:-1], hc_bg[:,3]) #np.sqrt(hc_ss.sum(axis=2)**2 + hc_bg**2))

calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', extra_panels=True)
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer', distance_units='rg', extra_panels=True)



In [ ]:
calc_and_plot_dadt(sam_newhard_data, fixedTime='outer',distance_units='rg')

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [1.0]:
    for rch in [100]:
        for nin in [0.0,0.5,1.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data,max_to_plot=20)
calc_and_plot_dadt(sam_data,distance_units='rg',max_to_plot=20)

In [ ]:
1e4*PC/(0.001*SPLC)/YR

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [1]:
    for rch in [10,100.0]:
        for nin in [-1.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data,max_to_plot=20)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [1.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-0.5]:
            for nout in [2.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+1.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=0.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=10.0, nu_in=-1.0, nu_out=+2.5)

calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
radii = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
radii = 10.0 ** radii
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    radii[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
#mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')

# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
print(f"{dadt.shape=}")


In [ ]:
##pta_dur = 16.03 * YR
#nfreqs = 40
##hifr = nfreqs/pta_dur
##pta_cad = 1.0 / (2 * hifr)
#fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
#fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)

nreals = 10
nloud = 1

freqs, freqs_edges = utils.pta_freqs()
gwb = sam.gwb(freqs, hard, realize=nreals, loudest=nloud, params=True)

In [ ]:
print(utils.stats(rads[-1,-1,-1,:]/dadt[-1,-1,-1,:]/GYR))
plt.xscale('log')
plt.yscale('log')
nskip = 1
cmap = plot._get_cmap('viridis')
colors = cmap(np.linspace(0, 1, int(sam.mtot.size/nskip)+1))
lw = np.arange(0,int(sam.mrat.size/nskip)+1, 0.1)

for i in np.arange(0,sam.mtot.size,nskip):
    for j in np.arange(0,sam.mrat.size,nskip):
        plt.plot(rads[i,j,0,:]/PC, -rads[i,j,0,:]/dadt[i,j,0,:]/GYR, 
                 alpha=0.5, color=colors[i], lw=lw[j])

In [ ]:
tt = times_evo[-1,-1,-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()

In [ ]:
nbins = [5, 10, 123, 0]
_, fit_lamp, fit_plaw, fit_med_lamp, fit_med_plaw = holo.librarian.fit_spectra(fobs_cents, gwb, nbins=nbins)

In [ ]:
num_snaps = len(nbins)
fig, axes = plt.subplots(figsize=[10, 5], ncols=2)
for med, fits, ax in zip([fit_med_lamp, fit_med_plaw], [fit_lamp, fit_plaw], axes):
    for ii, nn in enumerate(nbins):
        if np.all(fits[:, ii] == 0.0):
            continue
        color = ax._get_lines.get_next_color()
        kale.dist1d(fits[:, ii], ax=ax, label=str(nn), color=color)
        ax.axvline(med[ii], ls='--', color=color)
    
    ax.legend()
    
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = 1e-15 * np.power(xx, -2.0/3.0)
ax.plot(xx, yy, 'r-', alpha=0.5, lw=1.0, label="$10^{-15} \cdot f_\\mathrm{yr}^{-2/3}$")

fits = holo.librarian.get_gwb_fits_data(fobs_cents, gwb)

for ls, idx in zip([":", "--"], [1, -1]):
    med_lamp = fits['fit_med_lamp'][idx]
    med_plaw = fits['fit_med_plaw'][idx]
    yy = (10.0 ** med_lamp) * (xx ** med_plaw)
    label = fits['fit_nbins'][idx]
    label = 'all' if label in [0, None] else label
    ax.plot(xx, yy, color='k', ls=ls, alpha=0.5, lw=2.0, label=str(label) + " bins")

label = fits['fit_label'].replace(" | ", "\n")
fig.text(0.99, 0.99, label, fontsize=6, ha='right', va='top')

ax.legend()
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k:')

for nn in [5, 10, None]:
    xx, amp, gamma = holo.librarian.fit_powerlaw(fobs_cents, np.median(gwb, axis=-1), nn)
    ax.plot(xx, amp * (xx ** gamma), ls='--')

plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb, nsamp=None)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k-')

nreals = gwb.shape[1]

fits = np.zeros((nreals, 2))
for nn in range(nreals):
    yy = gwb[:, nn]
    xx, *fits[nn, :] = holo.librarian.fit_powerlaw(fobs_cents, yy, 5)
    cc, = ax.plot(xx, fits[nn, 0] * (xx ** fits[nn, 1]), ls='--', alpha=0.5)
    cc = cc.get_color()
    ax.plot(fobs_cents*YR, yy, color=cc, alpha=0.5)

plt.show()

draw_fits = fits.copy()
draw_fits[:, 0] = np.log10(draw_fits[:, 0])

kale.corner(draw_fits.T)
plt.show()


In [ ]:
hard_time=-2.2957907176750907
hard_gamma_inner=-1.3335554512862717
gsmf_phi0=-2.802178096487384
gsmf_mchar0=11.704311872442908
gsmf_alpha0=-1.7179504809027346
gpf_zbeta=2.397456708546681
gpf_qgamma=0.4609649227136603
gmt_norm=0.5765308121579338
gmt_zbeta=-0.26777937808636665
mmb_amp=8.301258575486393
mmb_plaw=0.4785954601355894
mmb_scatter=0.12386778329303819

hard_time = (10.0 ** hard_time) * GYR
gmt_norm = gmt_norm * GYR
mmb_amp = (10.0 ** mmb_amp) * MSOL

gsmf = holo.sam.GSMF_Schechter(phi0=gsmf_phi0, mchar0_log10=gsmf_mchar0, alpha0=gsmf_alpha0)
gpf = holo.sam.GPF_Power_Law(qgamma=gpf_qgamma, zbeta=gpf_zbeta)
gmt = holo.sam.GMT_Power_Law(time_norm=gmt_norm, zbeta=gmt_zbeta)
mmbulge = holo.host_relations.MMBulge_KH2013(mamp=mmb_amp, mplaw=mmb_plaw, scatter_dex=mmb_scatter)

sam = holo.sam.Semi_Analytic_Model(
    gsmf=gsmf, gpf=gpf, gmt=gmt, mmbulge=mmbulge,
    shape=20
)
hard = holo.hardening.Fixed_Time.from_sam(
    sam, hard_time, gamma_sc=hard_gamma_inner,
    progress=False
)
pta_dur = 16.03 * YR
nfreqs = 40
hifr = nfreqs/pta_dur
pta_cad = 1.0 / (2 * hifr)
fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)
gwb = sam.gwb(fobs_edges, realize=10, hard=hard)

plot.plot_gwb(fobs_cents, gwb)
plt.show()


In [ ]:
#SHAPE = None
SHAPE = 30
TIME = 1.0 * GYR

sam = holo.sam.Semi_Analytic_Model(shape=SHAPE)
hard = holo.hardening.Fixed_Time.from_sam(sam, TIME, interpolate_norm=False)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
rads = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
rads = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * rads
rads = 10.0 ** rads
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    rads[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
# Integrate (inverse) hardening rates to calculate total lifetime to each separation
times_evo = -utils.trapz_loglog(-1.0 / dadt, rads, axis=0, cumsum=True)


In [ ]:
tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()

In [ ]:
import numpy as np
test_alpha = np.arange(-1,1.1,0.1)
print(test_alpha)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(test_alpha, test_alpha, label='agw[rg]')
plt.plot(test_alpha, test_alpha+1, label='agw[pc]')
plt.plot(test_alpha, -3*test_alpha, label='dadt')
plt.plot(test_alpha, 4*test_alpha+1, label='a/dadt')
plt.plot([-1,1], [-2,-2], 'k', lw=4)
plt.plot([-1,1], [2,2], 'k', lw=4)
plt.plot([-1,1], [-1,-1], 'gray', lw=4)
plt.plot([-1,1], [1,1], 'gray',lw=4)
plt.plot([-2/3,-2/3],[-2,2], 'k')
plt.plot([0.25,0.25],[-2,2], 'k')
plt.plot([-1/3,-1/3],[-1,1], 'gray')
plt.plot([0,0],[-1,1], 'gray')
plt.plot([0.5,0.5],[-3,3], 'k:')
plt.legend()

In [ ]:
#suite_type = 'new_hardening_type0_toutvar'
suite_type = 'new_hardening_type0_foovar'
varied_values = dict(
            tout=[],
            nuivar=[],
            r9=[],
            alph=[]
        )
varName = [m for m in varied_values.keys() if m in suite_type]
print(f"{varName=}, {varied_values.keys()=}")
if len(varName)==1:         
    varName = varName[0]
else:
    print('blerg.')
print(varName)